**Vehicle Telemetry & Intelligent Fault Diagnosis System**
Developing a simulated vehicle telemetry pipeline that generates realistic vehicle operating data, injects configurable faults, and uses machine learning to identify abnormal behaviour and infer likely fault causes.

                    
                    
                    │      Driver      │
                    --------------------
                             |
                    ┌------------------┐
                    │ throttle/brake/  │
                    │ steering/gear    │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │       CAR        │
                    │                  │
                    │  ┌────────────┐  │
                    │  │   PARAMS   │  │
                    │  │ fixed      │  │
                    │  └────────────┘  │
                    │                  │
                    │  ┌────────────┐  │
                    │  │   STATE    │  │
                    │  │ changing   │  │
                    │  └────────────┘  │
                    │                  │
                    │ update_state()   │
                    └────────┬─────────┘
                             ↑
                    ┌────────┴─────────┐
                    │  FAULT INJECTOR  │
                    │ physical faults  │
                    └──────────────────┘
                             │
                             ↓
                    ┌──────────────────┐
                    │     SENSORS      │
                    │                  │
                    │ read(car.state)  │
                    │ + noise          │
                    │ + sensor faults  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │    TELEMETRY     │
                    │ timestamp +      │
                    │ sensor readings  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │   ML DIAGNOSER   │
                    │                  │
                    │ fault detection  │
                    │ + classification │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │     OUTPUT       │
                    │ fault +          │
                    │ confidence       │
                    └──────────────────┘

In [1]:
# Purpose: imports — random for faults/maneuvers, numpy for pi/clamps
#Importing packages
import random as rd
import numpy as np

| Class / Component     | Main purpose                                     | Parameters / Data it holds | Main methods                                                                               |
| --------------------- | ------------------------------------------------ | -------------------------- | ------------------------------------------------------------------------------------------ |
| **`Car`**             | Represents the vehicle                           | `params`, `state`          | `update_state()`                                                                           |
| **`Physics`**         | Calculates how the car physically behaves        | No permanent data required | `update()`, `calculate_acceleration()`, `calculate_rpm()`, `calculate_engine_temp()`, etc. |
| **`FaultInjector`**   | Creates vehicle faults                           | `self.faults`              | `apply()`, `reset()`                                                                       |
| **`SensorSystem`**    | Simulates the vehicle's sensors                  | Sensor noise/settings      | `read()`, `add_noise()`                                                                    |
| **`TelemetrySystem`** | Converts sensor readings into recorded telemetry | Sampling/settings          | `process()`, `save()`                                                                      |
| **`MLDiagnoser`**     | Learns to identify the fault from telemetry      | Trained ML model           | `train()`, `predict()`, `evaluate()`                                                       |
| **`simulation()`**    | Runs the entire experiment                       | `fault`, `duration`, `dt`  | Runs the classes in sequence                                                               |
| **`dataset`**         | Stores results from every run                    | Global dictionary          | No methods                                                                                 |


In [2]:
# Purpose: Car — owns data (parameters fixed, state evolving). No math inside.
#Lets first create a blueprint of a Car and then establish the parameters for telemetry.
class Car():
    def __init__(self):
        #parameters() - SI units throughout
        self.parameters = {
            "engine": {"idle_rpm": 800, "max_rpm": 6500, "max_torque": 200, "final_drive": 3.5, "gear_ratios": [3.2, 2.1, 1.5, 1.1, 0.85, 0.7]},
            "vehicle": {"mass": 1500, "wheelbase": 2.7, "track": 1.6, "CdA": 0.7, "rho": 1.225, "g": 9.81},
            "cooling": {"efficiency": 1.0, "tau": 120, "ambient": 25, "heat_k": 0.08},
            "brakes": {"max_torque": 3000, "cool_tau": 80, "heat_k": 50.0},
            "tyres": {"radius": 0.32, "rr_base": 0.015, "tau": 60, "heat_k": 0.5}
        }
        self.state = {"speed": 0, "rpm": 800, "gear": 1, "engine_temp": 25, "brake_temp": 25, "wheel_speed": 0}

    
    def update_state(self, dt, thr, brk, steer, physics):
        speed, gear, rpm, engT, brkT, tyreT, ws, slip, yaw = physics.update(self, dt, thr, brk, steer)
        self.state["speed"] = speed
        self.state["gear"] = gear
        self.state["rpm"] = rpm
        self.state["engine_temp"] = engT
        self.state["brake_temp"] = brkT
        self.state["tyre_temp"] = tyreT
        self.state["wheel_speed"] = ws
        self.state["slip"] = slip
        self.state["yaw"] = yaw

        '''
        car = Car()
        physics = Physics()
        driver = Driver()
        thr, brk, steer = driver.inputs(t)
        '''


        
        



In [3]:
# Purpose: Physics — owns all math. Reads car, returns new values.
class Physics:

    def update(self, car, dt, thr, brk, steer):
       ''' pre-load:
           car = Car()
           physics = Physics()
           driver = Driver()
           thr, brk, steer = driver.inputs(t)
       '''

       F_traction, F_brake, a = self.calcaccln(car, thr, brk)
       speed, gear = self.calcspeed(car, a, dt)
       rpm = self.calcrpm(car)
       engine_temp = self.calcengtemp(car, thr, dt)
       brake_temp = self.calcbraketemp(car, brk, dt)
       tyre_temp = self.calctyretemp(car, dt)
       ws, slip, yaw = self.calcwheels(car, steer)



       return speed, gear, rpm, engine_temp, brake_temp, tyre_temp, ws, slip, yaw

    #do thr, brk, steer = driver.inputs(t)
    def calcaccln(self,car, thr, brk):
        engine_torque = car.parameters["engine"]["max_torque"]
        gear = car.state["gear"]  # 1-6(5 gears), initial 1
        speed = car.state["speed"] #needed for F-drag

        ratios = car.parameters["engine"]["gear_ratios"]
        ratio = ratios[gear - 1]
        final_drive = car.parameters["engine"]["final_drive"]

        Rraw = car.parameters["tyres"]["radius"]
        Rmean = sum(Rraw.values()) / 4 if isinstance(Rraw, dict) else Rraw

        Traw = car.parameters["brakes"]["max_torque"]
        if isinstance(Rraw, dict) or isinstance(Traw, dict):
            # per-wheel after fault — expand singles to dicts for uniform math
            if not isinstance(Traw, dict): Traw = {w: Traw for w in ["FL", "FR", "RL", "RR"]}
            if not isinstance(Rraw, dict): Rraw = {w: Rraw for w in ["FL", "FR", "RL", "RR"]}
            F_brake = brk * sum(Traw[w] / Rraw[w] for w in ["FL", "FR", "RL", "RR"])
        else:
            F_brake = brk * 4 * Traw / Rraw
        

        F_traction = thr * engine_torque * ratio * final_drive / Rmean

        m = car.parameters["vehicle"]["mass"]
        rho = car.parameters["vehicle"]["rho"]
        CdA = car.parameters["vehicle"]["CdA"]
        rr = car.parameters["tyres"]["rr_base"]
        g = car.parameters["vehicle"]["g"]

        F_drag = 0.5 * rho * CdA * speed ** 2
        F_rr = rr * m * g

        a = (F_traction - F_brake - F_drag - F_rr) / m
        return F_traction, F_brake, a

    #do F_traction, F_brake, a = physics.calcaccln()

    def calcspeed(self, car, a, dt):
        speed = max(0, car.state["speed"]+ a*dt ) 
        gear = car.state["gear"]
        rpm = car.state["rpm"]
        if speed < 2: gear = 1
        else:
            if rpm > 5500 and gear < 6: gear += 1
            elif rpm < 1500 and gear >1: gear -= 1
        return speed, gear
        # do speed, gear = calcspeed(car, a, dt)
    
    def calcrpm(self, car):
        speed = car.state["speed"]
        gear = car.state["gear"]
        final_drive = car.parameters["engine"]["final_drive"]
        radius = car.parameters["tyres"]["radius"]
        ratios = car.parameters["engine"]["gear_ratios"]
        ratio = ratios[gear - 1]
        Rmean = sum(radius.values()) / 4 if isinstance(radius, dict) else radius

        rpm = speed * ratio * final_drive * 60 / (2* np.pi* Rmean)
        return max(800, min(6500, rpm))
    
    def calcengtemp(self, car, thr, dt):
        #T += ((k1×thr×rpm - eff×(T-amb)) / tau)×dt
        cool = car.parameters["cooling"]
        eff = cool["efficiency"]
        tau = cool["tau"]
        amb = cool["ambient"]
        k = cool["heat_k"]

        Te = car.state["engine_temp"]
        rpm = car.state["rpm"]

        Te += ((k*thr*rpm - eff*(Te-amb)) / tau) * dt
        return max(amb, min(150, Te))
    
    def calcbraketemp(self, car, brk, dt):
        #T += ((heat_k×brk×speed - (T-amb)) / cool_tau)×dt
        cool = car.parameters["cooling"]
        amb = cool["ambient"]
        brake = car.parameters["brakes"]
        tau = brake["cool_tau"]
        k = brake["heat_k"]
        speed = car.state["speed"]
        Tb = car.state["brake_temp"]
        if not isinstance(Tb, dict): Tb = {w: Tb for w in ["FL", "FR", "RL", "RR"]}
        
        Traw = car.parameters["brakes"]["max_torque"]
        Tmap = Traw if isinstance(Traw, dict) else {w: Traw for w in ["FL", "FR", "RL", "RR"]}
        Tmean = sum(Tmap.values()) / 4

        out = {}
        for w in ["FL", "FR", "RL", "RR"]:
            heat = k * (Tmap[w] / Tmean) * brk * speed
            out[w] = max(amb, min(400, Tb[w] + ((heat - (Tb[w] - amb)) / tau) * dt))
        return out
    
    def calctyretemp(self, car, dt):
        amb = car.parameters["cooling"]["ambient"]
        tau = car.parameters["tyres"]["tau"]
        k = car.parameters["tyres"]["heat_k"]
        speed = car.state["speed"]
        slip = car.state.get("slip", {w: 0.02 for w in ["FL","FR","RL","RR"]})
        if not isinstance(slip, dict):
            slip = {w: 0.02 for w in ["FL","FR","RL","RR"]}
        Tt = car.state.get("tyre_temp", {w: amb for w in ["FL","FR","RL","RR"]})
        if not isinstance(Tt, dict):
            Tt = {w: Tt for w in ["FL","FR","RL","RR"]}

        out = {}
        for w in ["FL","FR","RL","RR"]:
            heat = k * slip[w] * speed
            out[w] = max(amb, min(120, Tt[w] + ((heat - (Tt[w]-amb)) / tau) * dt))
        return out

    def calcwheels(self, car, steer):
        import math
        speed = car.state["speed"]
        wb = car.parameters["vehicle"]["wheelbase"]
        track = car.parameters["vehicle"]["track"]
        Rraw = car.parameters["tyres"]["radius"]
        R = Rraw if isinstance(Rraw, dict) else {w: Rraw for w in ["FL","FR","RL","RR"]}

        yaw = speed * math.tan(steer) / wb if wb else 0.0  # 0 straight
        v = {}
        ws = {}
        for w in ["FL","FR","RL","RR"]:
            lateral = -yaw * track/2 if "L" in w else yaw * track/2
            v[w] = speed + lateral
            ws[w] = v[w] / R[w] if R[w] else 0.0
        slip = {w: (ws[w]*R[w]-speed)/max(speed,2) for w in ws}
        return ws, slip, yaw
        

        










        

In [4]:
# Purpose: FaultInjector — choose once per run, alter params once at onset. Hidden from ML.
class initFault:
    #iinitalise a fault
    def __init__(self):
        self.faults = [None, "cooling_failure","brake_failure","engine_failure","tyre_degradation"]
        self.location = ["FR","FL","RR","RL"]
        self.active = None

    def choose(self):
        fault = rd.choice(self.faults)
        severity = round(rd.uniform(0.1,1.0), 2)

        label = "mild" if severity <= 0.3 else "moderate" if severity <= 0.7 else "severe"

        onset = round(rd.uniform(20, 40), 1)  

        if fault in ("brake_failure", "tyre_degradation"):
            loc = rd.choice(self.location)
        else:
            loc = None


        self.active = {
            "fault": fault, "severity": severity,
            "location": loc, "onset": onset, 
            "label": label
        }

        return self.active

    def apply(self, car):
        if self.active is None or self.active["fault"] is None:
            return
        f, s, loc = self.active["fault"], self.active["severity"], self.active["location"]

        if f == "cooling_failure":
            car.parameters["cooling"]["efficiency"] = max(0.1, 1.0 - 0.85 * s)
        elif f == "brake_failure":
            
            # needs per-wheel dict — convert once if still single value
            if not isinstance(car.parameters["brakes"].get("max_torque"), dict):
                base = car.parameters["brakes"]["max_torque"]
                car.parameters["brakes"]["max_torque"] = {w: base for w in ["FL","FR","RL","RR"]}
            car.parameters["brakes"]["max_torque"][loc] *= (1.0 - 0.85 * s)
        elif f == "engine_failure":
            car.parameters["engine"]["max_torque"] *= (1.0 - 0.5 * s)
        elif f == "tyre_degradation":
            if not isinstance(car.parameters["tyres"].get("radius"), dict):
                base_r = car.parameters["tyres"]["radius"]
                car.parameters["tyres"]["radius"] = {w: base_r for w in ["FL","FR","RL","RR"]}
                car.parameters["tyres"]["rr"] = {w: 0.015 for w in ["FL","FR","RL","RR"]}
            car.parameters["tyres"]["radius"][loc] *= (1.0 - 0.05 * s)
            car.parameters["tyres"]["rr"][loc] = 0.015 + 0.02 * s

In [5]:
# Purpose: Driver — input generator. Maneuver -> thr/brk/steer per timestep.
class Driver:
    def __init__(self, maneuver):
        self.maneuver = maneuver
        # per-run jitter, fixed for whole run
        self.thr_jit = rd.uniform(-0.05, 0.05)
        self.t_brake_start = rd.uniform(28, 35)  # for brake_stop/mixed

    def inputs(self, t): #t is current clock 0 --> duration
        thr = brk = 0.0
        steer = 0.0
        j = self.thr_jit

        if self.maneuver == "cruise_low":
            thr = 0.25 + j
        elif self.maneuver == "cruise_high":
            thr = 0.50 + j
        elif self.maneuver == "accel":
            thr = 0.2 + (0.6 * min(t / 20.0, 1.0)) + j
        elif self.maneuver == "brake_stop":
            thr = 0.4 + j
            if self.t_brake_start <= t <= self.t_brake_start + 5:
                thr, brk = 0.0, 0.8
        elif self.maneuver == "mixed":  # cruise-accel-cruise-brake
            if t < 20: thr = 0.25 + j
            elif t < 40: thr = 0.2 + 0.6 * ((t-20)/20.0) + j
            elif t < 60: thr = 0.4 + j
            elif t < 65: thr, brk = 0.0, 0.8
            else: thr = 0.4 + j

        return max(0,min(1,thr)), max(0,min(1,brk)), steer

In [6]:
maneuver = rd.choice(["cruise_low", "cruise_high", "accel", "brake_stop", "mixed"])

In [7]:
# Purpose: Telemetry — flat_row (dicts->columns) + run_once (one full run -> df+meta).
# new cell — simulation
def flat_row(t, thr, brk, steer, state):
    r = {"t": t, "thr": thr, "brk": brk, "steer": steer,
         "speed": state["speed"], "rpm": state["rpm"], "gear": state["gear"],
         "engine_temp": state["engine_temp"], "yaw": state.get("yaw", 0.0)}
    for w in ["FL","FR","RL","RR"]:
        bw = state["brake_temp"]
        tw = state.get("tyre_temp", {})
        ws = state["wheel_speed"]
        sl = state.get("slip", {})
        r[f"brake_{w}"] = bw[w] if isinstance(bw,dict) else bw
        r[f"tyre_{w}"] = tw[w] if isinstance(tw,dict) else tw
        r[f"ws_{w}"] = ws[w] if isinstance(ws,dict) else ws
        r[f"slip_{w}"] = sl[w] if isinstance(sl,dict) else sl
    return r

    
def run_once(maneuver=None, duration=120.0, dt=0.02):
    car = Car()
    physics = Physics()
    inj = initFault()
    sens = SensorSystem()
    active = inj.choose()  # fault, severity, location, onset, label

    if maneuver is None:
        maneuver = rd.choice(["cruise_low","cruise_high","accel","brake_stop","mixed"])
    driver = Driver(maneuver)

    applied = False
    rows = []
    t = 0.0
    while t <= duration:
        if not applied and t >= active["onset"]:
            inj.apply(car)
            applied = True
        thr, brk, steer = driver.inputs(t)
        car.update_state(dt, thr, brk, steer, physics)
        rows.append(sens.read(flat_row(t, thr, brk, steer, car.state)))
                    
        t += dt

    import pandas as pd
    df = pd.DataFrame(rows)
    return df, {"maneuver": maneuver, **active}

In [8]:
# Purpose: validation — forced fault (not random). lock_jit=True removes throttle noise; init_speed>0 gives rolling start for brake tests.
def run_forced(fault, location=None, severity=0.6, maneuver="cruise_low", duration=90, dt=0.05, lock_jit=True, init_speed=0):
    car, physics, inj = Car(), Physics(), initFault()
    inj.choose()
    inj.active.update({"fault": fault, "severity": severity, "location": location, "label": "moderate" if fault else "healthy", "onset": 30.0})
    car.state["speed"] = init_speed
    driver = Driver(maneuver)
    if lock_jit: driver.thr_jit = 0.0
    applied, rows, t = False, [], 0.0
    while t <= duration:
        if not applied and t >= 30.0:
            inj.apply(car); applied = True
        thr, brk, steer = driver.inputs(t)
        car.update_state(dt, thr, brk, steer, physics)
        rows.append(flat_row(t, thr, brk, steer, car.state))
        t += dt
    import pandas as pd
    return pd.DataFrame(rows)


In [9]:
# Purpose: SensorSystem — adds realistic sensor noise AFTER physics. ML trains on noisy telemetry, never clean state.
class SensorSystem:
    def __init__(self, speed_n=0.15, rpm_n=25, temp_n=0.5, ws_n=0.05):
        self.speed_n, self.rpm_n, self.temp_n, self.ws_n = speed_n, rpm_n, temp_n, ws_n
    def read(self, row):
        import numpy as np
        r = dict(row)
        r["speed"] += float(np.random.normal(0, self.speed_n))
        r["rpm"] += float(np.random.normal(0, self.rpm_n))
        r["engine_temp"] += float(np.random.normal(0, self.temp_n))
        for w in ["FL","FR","RL","RR"]:
            r[f"brake_{w}"] += float(np.random.normal(0, self.temp_n))
            r[f"tyre_{w}"] += float(np.random.normal(0, self.temp_n))
            r[f"ws_{w}"] += float(np.random.normal(0, self.ws_n))
        return r


In [10]:
# Purpose: validation gate — NORMAL vs faulty must diverge by eye before bulk dataset. Fails = fix heat_k, do not scale.
df_n = run_forced(None, None, 0.0, "cruise_low", 120, lock_jit=True)
df_f = run_forced("cooling_failure", None, 0.7, "cruise_low", 120, lock_jit=True)
print("NORMAL eng end:", round(float(df_n.engine_temp.iloc[-1]),1), "FAULT eng end:", round(float(df_f.engine_temp.iloc[-1]),1))
df_nb = run_forced(None, None, 0.0, "brake_stop", 120, lock_jit=True, init_speed=20)
df_fb = run_forced("brake_failure", "RR", 0.7, "brake_stop", 120, lock_jit=True, init_speed=20)
print("NORMAL brake_RR max:", round(float(df_nb.brake_RR.max()),1), "FAULT brake_RR max:", round(float(df_fb.brake_RR.max()),1))


NORMAL eng end: 80.6 FAULT eng end: 94.2
NORMAL brake_RR max: 40.4 FAULT brake_RR max: 33.5


In [11]:
# Purpose: Phase 1 dataset — n runs -> parquet + manifest.csv.
def build_dataset(n=5, duration=30.0, dt=0.05, outdir="data"):
    import os, pandas as pd
    os.makedirs(outdir, exist_ok=True)
    manifest = []
    for i in range(n):
        df, meta = run_once(None, duration, dt)  # random maneuver+fault
        df.to_parquet(f"{outdir}/run_{i:03d}.parquet", index=False)
        manifest.append({"run_id": f"{i:03d}", **meta})
    pd.DataFrame(manifest).to_csv(f"{outdir}/manifest.csv", index=False)
    return manifest

In [12]:
# Purpose: generate the 50-run dataset (50 x 120s @20Hz -> data/ + manifest.csv). Re-run to regenerate.
manifest = build_dataset(50, 120, 0.05)


In [13]:
# Purpose: dataset switch — all Phase 2/3 table builders read DATADIR. data=50-run pilot, data200=stratified bulk.
DATADIR = "data200"


In [14]:
# Purpose: Phase 2 windowing — parquet time-series -> ML samples. 5s windows / 2.5s stride @20Hz. Pre-onset windows = NORMAL even in faulty runs.
import numpy as np
def make_windows(run_id, df, meta, win_s=5.0, stride_s=2.5, hz=20):
    win, stride = int(win_s*hz), int(stride_s*hz)
    rows = []
    for s in range(0, len(df)-win+1, stride):
        w = df.iloc[s:s+win]
        t_end = float(w["t"].iloc[-1])
        fault = meta.get("fault")
        import math as _m
        if fault is None or (isinstance(fault, float) and _m.isnan(fault)): fault = None
        onset = float(meta.get("onset", 1e9))
        label = "NORMAL" if (fault is None or t_end < onset) else fault
        f = {"run_id": run_id, "t_start": float(w["t"].iloc[0]), "t_end": t_end, "label": label,
             "location": (lambda v: None if (v is None or (isinstance(v, float) and v != v)) else v)(meta.get("location")), "severity": meta.get("severity"), "maneuver": meta.get("maneuver")}
        for col in ["speed","rpm","engine_temp","thr","brk"]:
            f[f"{col}_mean"] = float(w[col].mean()); f[f"{col}_std"] = float(w[col].std()); f[f"{col}_max"] = float(w[col].max())
        f["eng_dT"] = float(w["engine_temp"].iloc[-1] - w["engine_temp"].iloc[0]) / win_s

        long_i = max(0, s + win - 1 - int(30 * hz))
        span = max((s + win - 1 - long_i) / hz, 5.0)
        f["eng_dT_long"] = float(df["engine_temp"].iloc[s + win - 1] - df["engine_temp"].iloc[long_i]) / span
        m15_i = max(0, s + win - 1 - int(15 * hz))
        f["eng_mean_15"] = float(df["engine_temp"].iloc[m15_i:s + win].mean())
        
        for wname in ["FL","FR","RL","RR"]:
            f[f"ws_{wname}_mean"] = float(w[f"ws_{wname}"].mean())
        f["ws_LR"] = (f["ws_FL_mean"]+f["ws_RL_mean"])/2 - (f["ws_FR_mean"]+f["ws_RR_mean"])/2
        f["ws_FR_diff"] = f["ws_FL_mean"] - f["ws_FR_mean"]
        for wname in ["FL", "FR", "RL", "RR"]:
            f[f"brake_{wname}_mean"] = float(w[f"brake_{wname}"].mean())
            f[f"brake_{wname}_max"] = float(w[f"brake_{wname}"].max())
            f[f"tyre_{wname}_mean"] = float(w[f"tyre_{wname}"].mean())
            f[f"slip_{wname}_max"] = float(w[f"slip_{wname}"].abs().max())
        f["brk_temp_spread"] = max(f[f"brake_{w}_mean"] for w in ["FL","FR","RL","RR"]) - min(f[f"brake_{w}_mean"] for w in ["FL","FR","RL","RR"])
        f["slip_max"] = max(f[f"slip_{w}_max"] for w in ["FL","FR","RL","RR"])
        gear_mode = float(w["gear"].mode().iloc[0])
        ratio = [3.2, 2.1, 1.5, 1.1, 0.85, 0.7][int(gear_mode) - 1]
        rpm_exp = float(w["speed"].mean()) * ratio * 3.5 * 60 / (2 * np.pi * 0.32)
        f["rpm_resid"] = float(w["rpm"].mean()) - rpm_exp
        _gap5 = w["speed"].values * ratio * 3.5 * 60 / (2 * np.pi * 0.32) - w["rpm"].values
        f["sag_persist"] = float((_gap5 > 200).mean())  # 4. engine family: persistence over 5s window
        f["deficit_int"] = float(np.maximum(_gap5, 0).mean() / 1000.0)  # 4. engine family: accumulated shortfall over 5s
        f["torque_proxy"] = float(_gap5.mean() / 1000.0)  # 4. engine family: mean lean over 5s
        _b0 = max(0, s + win - 1 - 600)
        _bh = df.iloc[_b0:s + win]
        _gr = int(_bh["gear"].mode().iloc[0])
        _ra = [3.2, 2.1, 1.5, 1.1, 0.85, 0.7][_gr - 1]
        _gap = _bh["speed"].values * _ra * 3.5 * 60 / (2 * np.pi * 0.32) - _bh["rpm"].values
        f["sag30"] = float((_gap > 200).mean())  # 4. engine family: persistence over 30s lookback
        f["def30"] = float(np.maximum(_gap, 0).mean() / 1000.0)  # 4. engine family: accumulated shortfall (severity fuel)
        f["prox30"] = float(_gap.mean() / 1000.0)  # 4. engine family: mean lean, direction+depth
        rows.append(f)
    import pandas as pd
    return pd.DataFrame(rows)

# Demo: 1 NORMAL-ish + 1 faulty run -> window counts
import pandas as pd
m = pd.read_csv(DATADIR + "/manifest.csv")
demo = []
for _, r in m.head(4).iterrows():
    df = pd.read_parquet(f"{DATADIR}/run_{int(r['run_id']):03d}.parquet")
    demo.append(make_windows(r['run_id'], df, r.to_dict()))
import pandas as _pd
demo_df = _pd.concat(demo, ignore_index=True)
print(demo_df["label"].value_counts().to_string())
print("windows:", len(demo_df), "features:", len(demo_df.columns))


In [15]:
# Purpose: Phase 2 anomaly — IsolationForest trained on NORMAL windows only. Flags ABNORMAL incl. unseen faults.
def build_window_table(n_runs=50):
    import pandas as pd
    m = pd.read_csv(DATADIR + "/manifest.csv").head(n_runs)
    parts = []
    for _, r in m.iterrows():
        df = pd.read_parquet(f"{DATADIR}/run_{int(r['run_id']):03d}.parquet")
        parts.append(make_windows(r['run_id'], df, r.to_dict()))
    import pandas as _pd
    return _pd.concat(parts, ignore_index=True)

def train_anomaly(df, feat_cols):
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    tr = df[df.label=="NORMAL"]
    sc = StandardScaler().fit(tr[feat_cols])
    cl = IsolationForest(n_estimators=200, contamination="auto", random_state=0).fit(sc.transform(tr[feat_cols]))
    return sc, cl

FEATS = [c for c in ["speed_mean","rpm_mean","engine_temp_mean","eng_dT","ws_LR","ws_FR_diff","thr_mean","brk_mean","brake_FL_mean", "brake_FR_mean", "brake_RL_mean", "brake_RR_mean", "brk_temp_spread", "tyre_FL_mean", "slip_max", "rpm_resid", "eng_dT_long", "eng_mean_15"] ]
W = build_window_table(12)
SC, CL = train_anomaly(W, FEATS)
import numpy as np
pred = CL.predict(SC.transform(W[FEATS]))  # 1 normal, -1 abnormal
W["pred_ab"] = (pred==-1).astype(int)
print("train windows:", len(W), "NORMAL:", int((W.label=="NORMAL").sum()))
print(W.groupby("label")["pred_ab"].mean().round(3).to_string())

train windows: 564 NORMAL: 564
label
NORMAL    0.131


In [16]:
for lo, hi, name in [(0.1, 0.2, "mild"), (0.4, 0.6, "moderate"), (0.8, 1.0, "severe")]:
    sub = W[(W["severity"] >= lo) & (W["severity"] <= hi)]
    print(name, len(sub))
    print(sub.groupby("label")["pred_ab"].mean().round(3).to_string())

W["severity"].head(10)
W["run_id"].value_counts()

mild 0
Series([], )
moderate 0
Series([], )
severe 0
Series([], )


run_id
0     47
1     47
2     47
3     47
4     47
5     47
6     47
7     47
8     47
9     47
10    47
11    47
Name: count, dtype: int64

In [17]:
# Purpose: Phase 2 fault classifier (cML) — RandomForest on ABNORMAL windows only. Group split by run (no leakage).
def build_full_table():
    import pandas as pd
    m = pd.read_csv(DATADIR + "/manifest.csv")
    parts = []
    for _, r in m.iterrows():
        df = pd.read_parquet(f"{DATADIR}/run_{int(r['run_id']):03d}.parquet")
        parts.append(make_windows(r['run_id'], df, r.to_dict()))
    import pandas as _pd
    return _pd.concat(parts, ignore_index=True)

FULL = build_full_table()
AB = FULL[FULL.label!="NORMAL"].copy()
# 1. proxy drop: remove driver-demand fingerprints (thr/brk/speed/steer) so the model diagnoses from vehicle response, not maneuver identity
PROXY = ("thr_","brk_","speed_","steer")
CFEATS = [c for c in FULL.columns if c not in ("run_id","label","location","severity","maneuver","t_start","t_end") and not c.startswith(PROXY)]  # drop driver-demand proxies: diagnose from response, not maneuver
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
gss = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=0).split(AB[CFEATS], AB["label"], AB["run_id"])
tri, tei = next(gss)
TRF = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(AB[CFEATS].iloc[tri], AB["label"].iloc[tri])
print("abnormal windows:", len(AB), "train:", len(tri), "test:", len(tei))
from sklearn.metrics import accuracy_score, confusion_matrix
pr = TRF.predict(AB[CFEATS].iloc[tei])
print("type accuracy:", round(accuracy_score(AB["label"].iloc[tei], pr), 3))
print("labels:", list(TRF.classes_))
print(confusion_matrix(AB["label"].iloc[tei], pr))


In [ ]:
# 3. calibration + abstention (canon step 3): confidence must track accuracy; report-if-confident policy.
P = TRF.predict_proba(AB[CFEATS].iloc[tei])
conf = P.max(1)
pred = TRF.classes_[P.argmax(1)]
true = AB["label"].iloc[tei].values
print("base acc:", round((pred == true).mean(), 3))
for lo in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    hi = lo + 0.1 if lo < 0.9 else 1.01
    m = (conf >= lo) & (conf < hi)
    if m.sum():
        print(f"{lo:.1f}-{hi:.2f}: acc={((pred[m] == true[m]).mean()):.3f} n={m.sum()}")
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    print(f"t={t}: coverage={m.mean():.3f} acc={((pred[m] == true[m]).mean() if m.sum() else float('nan')):.3f}")


In [18]:
# Purpose: Phase 3 localisation — predict wheel (FL/FR/RL/RR) or system (cooling/engine). Trained on abnormal windows with true type as extra feat.
LOCT = AB.copy()
LOCT["loc"] = LOCT.apply(lambda r: r["location"] if r["label"] in ("brake_failure","tyre_degradation") else "system", axis=1)
LOCT["pred_type"] = TRF.predict(AB[CFEATS])
LOCD = __import__("pandas").get_dummies(LOCT["pred_type"], prefix="pt")
LOCF = CFEATS + list(LOCD.columns)
LOCT = __import__("pandas").concat([LOCT, LOCD], axis=1)
from sklearn.model_selection import GroupShuffleSplit as _G
from sklearn.ensemble import RandomForestClassifier as _R
g2 = _G(n_splits=1, test_size=0.25, random_state=1).split(LOCT[LOCF], LOCT["loc"], LOCT["run_id"])
a, b = next(g2)
LOC_COLS = list(LOCD.columns)
LRF = _R(n_estimators=300, min_samples_leaf=5, random_state=1, n_jobs=-1).fit(LOCT[LOCF].iloc[a], LOCT["loc"].iloc[a])
from sklearn.metrics import accuracy_score as _a
print("loc accuracy:", round(_a(LOCT["loc"].iloc[b], LRF.predict(LOCT[LOCF].iloc[b])), 3))
print("classes:", list(LRF.classes_))


loc accuracy: 0.722
classes: ['FL', 'FR', 'RL', 'RR', 'system']


In [19]:
# Purpose: Phase 3 severity — RandomForest regressor on abnormal windows -> numeric severity_s. Group split by run.
SEV = AB.copy()
from sklearn.model_selection import StratifiedGroupKFold as _S
from sklearn.ensemble import RandomForestRegressor as _R
import numpy as _np
SEV["bucket"] = _np.where(SEV["severity"]<=0.33, 0, _np.where(SEV["severity"]<=0.66, 1, 2))
gs = _S(n_splits=4, shuffle=True, random_state=2).split(SEV[CFEATS], SEV["bucket"], SEV["run_id"])
sa, sb = next(gs)
SRF = _R(n_estimators=300, min_samples_leaf=5, random_state=2, n_jobs=-1).fit(SEV[CFEATS].iloc[sa], SEV["severity"].iloc[sa])
from sklearn.metrics import mean_absolute_error as _m
pr = SRF.predict(SEV[CFEATS].iloc[sb])
print("severity MAE:", round(_m(SEV["severity"].iloc[sb], pr), 3))
import pandas as _p
_r = _p.DataFrame({"true": SEV["severity"].iloc[sb].values, "pred": pr, "fault": SEV["label"].iloc[sb].values})
print(_r.groupby("fault").apply(lambda g: round(_m(g["true"], g["pred"]), 3)).to_string())


severity MAE: 0.158
fault
brake_failure       0.232
cooling_failure     0.212
engine_failure      0.114
tyre_degradation    0.081


/var/folders/nx/30hzfndn1qv9997_rx8zt7680000gn/T/ipykernel_984/3083331882.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(_r.groupby("fault").apply(lambda g: round(_m(g["true"], g["pred"]), 3)).to_string())


In [20]:
# Purpose: stratified bulk data (200 runs: 5 faults x 5 maneuvers x 8 each) -> data200/ + manifest. Severity/location/onset random inside. Takes ~10-15 min on i7.
def build_bulk_dataset(n_per_combo=8, duration=120.0, dt=0.05, outdir="data200"):
    import os, pandas as pd
    os.makedirs(outdir, exist_ok=True)
    faults = [None, "cooling_failure", "brake_failure", "engine_failure", "tyre_degradation"]
    maneuvers = ["cruise_low", "cruise_high", "accel", "brake_stop", "mixed"]
    manifest, i = [], 0
    for f in faults:
        for man in maneuvers:
            for _ in range(n_per_combo):
                car, physics, inj = Car(), Physics(), initFault()
                inj.choose()
                inj.active["fault"] = f
                from math import isnan as _n
                inj.active["location"] = __import__("random").choice(["FL","FR","RL","RR"]) if f in ("brake_failure","tyre_degradation") else None
                if f is None: inj.active["severity"], inj.active["label"] = 0.0, "healthy"
                driver = Driver(man)
                applied, rows, t = False, [], 0.0
                sens = SensorSystem()
                while t <= duration:
                    if not applied and t >= inj.active["onset"]:
                        inj.apply(car); applied = True
                    thr, brk, steer = driver.inputs(t)
                    car.update_state(dt, thr, brk, steer, physics)
                    rows.append(sens.read(flat_row(t, thr, brk, steer, car.state)))
                    t += dt
                df = pd.DataFrame(rows)
                df.to_parquet(f"{outdir}/run_{i:03d}.parquet", index=False)
                manifest.append({"run_id": f"{i:03d}", "maneuver": man, **inj.active})
                i += 1
    pd.DataFrame(manifest).to_csv(f"{outdir}/manifest.csv", index=False)
    return manifest

# smoke: 1 per combo = 25 quick short runs into bulkrun_smoke/
# smoke = build_bulk_dataset(1, 20, 0.05, "data_smoke")


In [21]:
bulk = build_bulk_dataset()

In [22]:
# Purpose: generalisation evaluation — leave-one-fault-out (novelty) + unseen-maneuver. Caches windows table to windows_DATADIR.parquet.
import os as _os, pandas as _pd
WTAB = DATADIR + "/windows.parquet"
if _os.path.exists(WTAB):
    EV = _pd.read_parquet(WTAB)
else:
    _m = _pd.read_csv(DATADIR + "/manifest.csv")
    _p = []
    for _, _r in _m.iterrows():
        _d = _pd.read_parquet(f"{DATADIR}/run_{int(_r['run_id']):03d}.parquet")
        _p.append(make_windows(_r['run_id'], _d, _r.to_dict()))
    EV = _pd.concat(_p, ignore_index=True)
    EV.to_parquet(WTAB, index=False)
print("EV windows:", len(EV))

PROXY = ("thr_","brk_","speed_","steer")
CFEATS = [c for c in EV.columns if c not in ("run_id","label","location","severity","maneuver","t_start","t_end") and not c.startswith(PROXY)]
# A. leave-one-fault-out: anomaly must flag held-out fault as novel
from sklearn.ensemble import IsolationForest as _I
from sklearn.preprocessing import StandardScaler as _S
for _held in ["brake_failure", "cooling_failure", "engine_failure", "tyre_degradation"]:
    _tr = EV[(EV.label=="NORMAL") & (~EV.run_id.isin(EV[EV.label==_held].run_id.unique()))]
    _te = EV[EV.label==_held]
    _sc = _S().fit(_tr[CFEATS])
    _cl = _I(n_estimators=200, contamination="auto", random_state=0).fit(_sc.transform(_tr[CFEATS]))
    _flag = (_cl.predict(_sc.transform(_te[CFEATS]))==-1).mean()
    print(f"hold {_held}: novel-flag rate {round(float(_flag),3)} (n={len(_te)})")


EV windows: 9400
hold brake_failure: novel-flag rate 0.389 (n=1462)
hold cooling_failure: novel-flag rate 0.448 (n=1438)
hold engine_failure: novel-flag rate 0.345 (n=1450)
hold tyre_degradation: novel-flag rate 0.547 (n=1451)


In [23]:
# Purpose: maneuver-conditioned residuals — subtract healthy same-maneuver median so faults read as deviation, not regime.
BASE = EV[EV.label=="NORMAL"].groupby("maneuver")[CFEATS].median(numeric_only=True)
RFEATS = [c+"_r" for c in CFEATS]
def add_resid(df):
    import pandas as _p
    r = df.copy()
    for c in CFEATS:
        r[c+"_r"] = df[c].values - df["maneuver"].map(BASE[c]).values
    return r
EVR = add_resid(EV)
print("resid cols:", len(RFEATS))


resid cols: 33


In [24]:
# Purpose: re-score everything on residual feats — resub anomaly, LOFO novelty, unseen-maneuver.
from sklearn.ensemble import IsolationForest as _I2
from sklearn.preprocessing import StandardScaler as _S2
_tr = EVR[EVR.label=="NORMAL"]
_sc = _S2().fit(_tr[RFEATS])
_cl = _I2(n_estimators=200, contamination="auto", random_state=0).fit(_sc.transform(_tr[RFEATS]))
_pr = (_cl.predict(_sc.transform(EVR[RFEATS]))==-1).astype(int)
import pandas as _p2
_o = _p2.DataFrame({"label": EVR["label"].values, "pred": _pr})
print("resub:")
print(_o.groupby("label")["pred"].mean().round(3).to_string())
for _held in ["brake_failure","cooling_failure","engine_failure","tyre_degradation"]:
    _a = EVR[(EVR.label=="NORMAL") & (~EVR.run_id.isin(EVR[EVR.label==_held].run_id.unique()))]
    _b = EVR[EVR.label==_held]
    _s = _S2().fit(_a[RFEATS])
    _c = _I2(n_estimators=200, contamination="auto", random_state=0).fit(_s.transform(_a[RFEATS]))
    print(f"lofo {_held}: {round(float((_c.predict(_s.transform(_b[RFEATS]))==-1).mean()),3)}")
for _hm in ["cruise_low","cruise_high","accel","brake_stop","mixed"]:
    _a = EVR[(EVR.label=="NORMAL") & (EVR.maneuver!=_hm)]
    _n = EVR[(EVR.label=="NORMAL") & (EVR.maneuver==_hm)]
    _f = EVR[(EVR.label!="NORMAL") & (EVR.maneuver==_hm)]
    _s = _S2().fit(_a[RFEATS])
    _c = _I2(n_estimators=200, contamination="auto", random_state=0).fit(_s.transform(_a[RFEATS]))
    print(f"man {_hm}: FPR {round(float((_c.predict(_s.transform(_n[RFEATS]))==-1).mean()),3)} det {round(float((_c.predict(_s.transform(_f[RFEATS]))==-1).mean()),3)}")


resub:
label
NORMAL              0.181
brake_failure       0.385
cooling_failure     0.433
engine_failure      0.352
tyre_degradation    0.502
lofo brake_failure: 0.363
lofo cooling_failure: 0.39
lofo engine_failure: 0.302
lofo tyre_degradation: 0.497
man cruise_low: FPR 0.012 det 0.094
man cruise_high: FPR 0.086 det 0.35
man accel: FPR 0.34 det 0.589
man brake_stop: FPR 0.439 det 0.943
man mixed: FPR 0.304 det 0.809


In [25]:
# Purpose: per-maneuver detectors — one IsoF per maneuver on raw feats (regime keeps absolute info, boundary adapts).
from sklearn.ensemble import IsolationForest as _I3
from sklearn.preprocessing import StandardScaler as _S3
import pandas as _p3
_rows = []
for _hm in ["cruise_low","cruise_high","accel","brake_stop","mixed"]:
    _tr = EV[(EV.label=="NORMAL") & (EV.maneuver==_hm)]
    _te = EV[EV.maneuver==_hm]
    _s = _S3().fit(_tr[CFEATS])
    _c = _I3(n_estimators=200, contamination="auto", random_state=0).fit(_s.transform(_tr[CFEATS]))
    _pr = (_c.predict(_s.transform(_te[CFEATS]))==-1).astype(int)
    _o = _p3.DataFrame({"label": _te["label"].values, "pred": _pr})
    print(_hm, "n=", len(_te))
    print(_o.groupby("label")["pred"].mean().round(3).to_string())
    _rows.append(_o.assign(man=_hm))
_ALL = _p3.concat(_rows, ignore_index=True)
print("overall:")
print(_ALL.groupby("label")["pred"].mean().round(3).to_string())


cruise_low n= 1880
label
NORMAL              0.141
brake_failure       0.225
cooling_failure     0.360
engine_failure      0.171
tyre_degradation    0.524
cruise_high n= 1880
label
NORMAL              0.182
brake_failure       0.158
cooling_failure     0.124
engine_failure      0.247
tyre_degradation    0.356
accel n= 1880
label
NORMAL              0.194
brake_failure       0.138
cooling_failure     0.152
engine_failure      0.300
tyre_degradation    0.453
brake_stop n= 1880
label
NORMAL              0.290
brake_failure       0.639
cooling_failure     0.738
engine_failure      0.647
tyre_degradation    0.813
mixed n= 1880
label
NORMAL              0.310
brake_failure       0.841
cooling_failure     0.865
engine_failure      0.837
tyre_degradation    0.912
overall:
label
NORMAL              0.223
brake_failure       0.399
cooling_failure     0.446
engine_failure      0.440
tyre_degradation    0.610


In [26]:
# Purpose: Phase 4 diagnostic report — per-run evidence from features (thresholds, not LLM prose).
def diagnose_run(run_id):
    import pandas as _p, collections as _c
    df = _p.read_parquet(f"{DATADIR}/run_{int(run_id):03d}.parquet")
    man = _p.read_csv(DATADIR + "/manifest.csv")
    meta = man[man.run_id==int(run_id)].iloc[0].to_dict()
    W = make_windows(run_id, df, meta)
    _fl = meta.get("fault")
    _on = float(meta.get("onset", 0))
    post = W[W.t_end>=_on] if _fl is not None and str(_fl)!="nan" else W
    _af = (CL.predict(SC.transform(W[FEATS]))==-1)
    status = "ABNORMAL" if _af.mean()>=0.08 else "NORMAL"  # calibrated: NORMAL≈0.06, cruise_low faults 0.09+; mild-cruise blind spot documented
    ab = TRF.predict(post[CFEATS])
    top, n = _c.Counter(ab).most_common(1)[0]
    import pandas as _q
    _pt = _q.get_dummies(_p.Series(TRF.predict(post[CFEATS])), prefix="pt").reindex(columns=LOC_COLS, fill_value=0)
    _pf = _p.concat([post[CFEATS].reset_index(drop=True), _pt.reset_index(drop=True)], axis=1)
    loctop = _c.Counter(LRF.predict(_pf[LOCF])).most_common(1)[0][0]
    sev = float(SRF.predict(post[CFEATS]).mean())
    ev = ["eng_dT_long max %.2f C/s" % W["eng_dT_long"].max(), "brake spread max %.1f C" % W["brk_temp_spread"].max(), "rpm_resid min %.0f rpm" % W["rpm_resid"].min(), "slip max %.3f" % W["slip_max"].max()]
    return {"run": int(run_id), "status": status, "fault": (top if status=="ABNORMAL" else "none"), "conf": round(n/len(ab),2), "location": loctop, "severity_s": round(sev,2), "evidence": ev, "truth": _fl}

print(diagnose_run(0))
print(diagnose_run(40))


{'run': 0, 'status': 'ABNORMAL', 'fault': 'brake_failure', 'conf': 0.79, 'location': 'system', 'severity_s': 0.54, 'evidence': ['eng_dT_long max 0.60 C/s', 'brake spread max 0.2 C', 'rpm_resid min -1364 rpm', 'slip max 0.000'], 'truth': nan}
{'run': 40, 'status': 'ABNORMAL', 'fault': 'cooling_failure', 'conf': 0.68, 'location': 'system', 'severity_s': 0.54, 'evidence': ['eng_dT_long max 0.50 C/s', 'brake spread max 0.3 C', 'rpm_resid min -714 rpm', 'slip max 0.000'], 'truth': 'cooling_failure'}


Explanding this a DLM (1D CNN)

In [27]:
from torch import nn, conv1d, tensor, stack
import torch
import pandas as pd
import numpy as np
import re
import glob

In [28]:
signal_cols = (["speed", "rpm", "engine_temp"] + [f"brake_{w}" for w in ["FL", "FR", "RL", "RR"]] + [f"tyre_{w}" for w in ["FL", "FR", "RL", "RR"]] + [f"ws_{w}" for w in ["FL", "FR", "RL", "RR"]] + [f"slip_{w}" for w in ["FL", "FR", "RL", "RR"]])
mats =[]
rids=[]
ids, y = {"cooling_failure": 0, "brake_failure": 1, "engine_failure": 2, "tyre_degradation": 3}, [] #label vec ids
mdf = pd.read_csv("data200/manifest.csv")

for i in range(0, 200): 
    df = pd.read_parquet(f"data200/run_{i:03d}.parquet")
    row = mdf[mdf.run_id == i].iloc[0]
    

    for s in range(0, len(df)-100, 50): #0 to 2300 (-100 to leave space for the final window, the last pointer has to be 2300 as 2401 is the length)
        t_start = float(df["t"].iloc[s])
        t_end = float(df["t"].iloc[s+99])
        onset = row["onset"]
        if pd.isna(row["fault"]) or t_end < onset : continue
        if t_start < onset and onset < t_end: continue #keeping the fault growing windows out of it.
        mats.append(df.iloc[s:s+100][signal_cols].values.T)
        y.append(ids[row["fault"]])
        rids.append(i)
   

X = torch.tensor(np.stack(mats), dtype = torch.float32)
Y = torch.tensor(y, dtype= torch.long)
vrids = np.asarray(rids)

print(X.shape, X.dtype)  
print(Y.shape, Y.dtype)  
#m



torch.Size([5491, 19, 100]) torch.float32
torch.Size([5491]) torch.int64


In [ ]:
#splitting it

from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=1)
for tr_id, te_id in cv.split(X, Y, groups=vrids):
    xtr, xte = X[tr_id], X[te_id]
    ytr, yte = Y[tr_id], Y[te_id]
    
xtr.shape, xte.shape, ytr.shape, yte.shape

(torch.Size([4107, 19, 100]),
 torch.Size([1384, 19, 100]),
 torch.Size([4107]),
 torch.Size([1384]))

In [30]:
#further processing before making the actual network
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(xtr, ytr)
test_ds = TensorDataset(xte, yte)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [31]:
#creating the CNN

class CNN(nn.Module):
    def __init__(self, in_channel = 19, n_cls = 4):
        super().__init__()

        self.cnn= nn.Sequential(
            nn.Conv1d(in_channel, 16, 7, padding = 3), nn.BatchNorm1d(16), nn.ReLU(), nn.AvgPool1d(2),
            nn.Conv1d(16, 32, 5, padding = 2), nn.BatchNorm1d(32), nn.ReLU(), nn.AdaptiveAvgPool1d(1), 
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(32, n_cls)
        )

    def forward(self, x):
        return self.cnn(x)


In [32]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr =0.001)

def one_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in train_loader:

        optimizer.zero_grad() #clear the gradients

        predn = model(x_batch)
        loss = criterion(predn, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x_batch.size(0)
    
    return total_loss / len(train_loader.dataset)

In [33]:
epochs = 15

for epoch in range(epochs):
    avg_loss = one_epoch(model, train_loader, criterion, optimizer)

    print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")



Epoch 1/15 | Loss: 1.3784
Epoch 2/15 | Loss: 1.2817
Epoch 3/15 | Loss: 1.2221
Epoch 4/15 | Loss: 1.1874
Epoch 5/15 | Loss: 1.1656
Epoch 6/15 | Loss: 1.1639
Epoch 7/15 | Loss: 1.1639
Epoch 8/15 | Loss: 1.1300
Epoch 9/15 | Loss: 1.1281
Epoch 10/15 | Loss: 1.1344
Epoch 11/15 | Loss: 1.1516
Epoch 12/15 | Loss: 1.1146
Epoch 13/15 | Loss: 1.1291
Epoch 14/15 | Loss: 1.1084
Epoch 15/15 | Loss: 1.1026


In [34]:
def evaluate(model, test_loader, criterion):
    model.eval()
    total_loss = 0.0
    correct =0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            predn = model(x_batch)
            loss = criterion(predn, y_batch)

            total_loss += loss.item() * x_batch.size(0)

            prediction = predn.argmax(dim=1)
            correct += (prediction == y_batch).sum().item()
            total += y_batch.size(0)
        
        return total_loss/ total, correct / total 

In [35]:
tloss, acc = evaluate(model, test_loader, criterion)

print(f"test loss: {tloss}, accuracy: {acc}")

test loss: 1.4493329366507557, accuracy: 0.2745664739884393


In [36]:
import numpy as np

print("Train:", np.unique(ytr.numpy(), return_counts=True))
print("Test:", np.unique(yte.numpy(), return_counts=True))

from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, zero_division=0))

Train: (array([0, 1, 2, 3]), array([ 949, 1139, 1056,  963]))
Test: (array([0, 1, 2, 3]), array([462, 204, 312, 406]))
[[125 175  61 101]
 [ 71  64  23  46]
 [ 66  46 146  54]
 [ 83 215  63  45]]
              precision    recall  f1-score   support

           0       0.36      0.27      0.31       462
           1       0.13      0.31      0.18       204
           2       0.50      0.47      0.48       312
           3       0.18      0.11      0.14       406

    accuracy                           0.27      1384
   macro avg       0.29      0.29      0.28      1384
weighted avg       0.31      0.27      0.28      1384



In [37]:
#normalising xtr and xte

mean = xtr.mean(dim = (0,2), keepdim=True)
std = xtr.std(dim=(0,2), keepdim=True).clamp(min = 1e-6)

xtr = (xtr - mean) / std
xte = (xte - mean) / std

test_loader_normalised = DataLoader(TensorDataset(xte, yte), batch_size=64, shuffle=False)
train_loader_normalised = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)

print("Train:", xtr.shape, xtr.mean().item(), xtr.std().item())
print("Test:", xte.shape)

Train: torch.Size([4107, 19, 100]) 1.9062448473050608e-06 0.8885222673416138
Test: torch.Size([1384, 19, 100])


In [38]:
MODEL = CNN()
optimizer3=torch.optim.Adam(MODEL.parameters(), lr =0.001)
criterion3=nn.CrossEntropyLoss()

ep = 20
for i in range(ep):
    avl=one_epoch(MODEL, train_loader_normalised, criterion3, optimizer3)
    print(f"{i+1}/{ep}, loss: {avl}")

1/20, loss: 1.320441816071974
2/20, loss: 1.1899930277328175
3/20, loss: 1.1532008259842685
4/20, loss: 1.1282306616690028
5/20, loss: 1.1102802766510131
6/20, loss: 1.098203438112618
7/20, loss: 1.0999478489951493
8/20, loss: 1.0828046756908787
9/20, loss: 1.0791024538557519
10/20, loss: 1.0769243907951687
11/20, loss: 1.074201610653192
12/20, loss: 1.067542736681819
13/20, loss: 1.0592066090502297
14/20, loss: 1.0553802314511698
15/20, loss: 1.0492627479050727
16/20, loss: 1.0360597192965373
17/20, loss: 1.0410319102022345
18/20, loss: 1.047940918671873
19/20, loss: 1.049687659360209
20/20, loss: 1.0294037960090572


In [39]:
tloss, acc = evaluate(
    MODEL,
    test_loader_normalised,
    criterion3
)

print(f"Test loss: {tloss:.4f}")
print(f"Test accuracy: {acc:.2%}")

Test loss: 1.3138
Test accuracy: 34.47%


In [40]:
from sklearn.metrics import confusion_matrix, classification_report

MODEL.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader_normalised:
        logits = MODEL(xb)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(
    all_labels, all_preds, zero_division=0
))

[[166 108  56 132]
 [ 51  77  19  57]
 [ 45  21 171  75]
 [106 230   7  63]]
              precision    recall  f1-score   support

           0       0.45      0.36      0.40       462
           1       0.18      0.38      0.24       204
           2       0.68      0.55      0.61       312
           3       0.19      0.16      0.17       406

    accuracy                           0.34      1384
   macro avg       0.37      0.36      0.35      1384
weighted avg       0.39      0.34      0.36      1384



In [42]:
print("Channel means:", xtr.mean(dim=(0, 2)))
print("Channel stds:", xtr.std(dim=(0, 2)))
for i, s in enumerate(xtr.std(dim=(0, 2))):
    print(i, s.item())

Channel means: tensor([ 3.5059e-06, -1.0888e-07, -1.3170e-07,  6.3050e-06, -3.1941e-07,
        -9.2935e-07,  1.2387e-05,  6.2165e-05,  1.3398e-06,  1.8273e-05,
        -7.8727e-05,  3.8026e-06, -5.1076e-07,  5.0087e-06,  4.4146e-06,
         7.5782e-19,  3.7127e-20, -7.2151e-19,  9.0836e-19])
Channel stds: tensor([1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 4.6823e-11, 4.6783e-11, 4.6763e-11,
        4.6745e-11])
0 1.0
1 1.0
2 1.0
3 1.0
4 0.9999999403953552
5 1.0
6 1.0
7 1.0
8 1.0
9 0.9999999403953552
10 1.0
11 0.9999999403953552
12 0.9999999403953552
13 0.9999999403953552
14 1.0
15 4.682297258251644e-11
16 4.67830288397586e-11
17 4.6763468097843486e-11
18 4.674504533452861e-11


Try-3, this time we will remove the 4 channels : slip_* as their std devn was around 5e-11, basically 0.

In [43]:
#splitting data again for fresh xte/xtr and y...:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=1)
for tr_id, te_id in cv.split(X, Y, groups=vrids):
    xtr3, xte3 = X[tr_id], X[te_id]
    ytr3, yte3 = Y[tr_id], Y[te_id]
    
xtr3.shape, xte3.shape, ytr.shape, yte.shape

(torch.Size([4107, 19, 100]),
 torch.Size([1384, 19, 100]),
 torch.Size([4107]),
 torch.Size([1384]))

In [ ]:
#keeping only the non slip_* channels
keep = [i for i, col in enumerate(signal_cols)
        if not col.startswith("slip_")]

Xtr3 = xtr3[:, keep, :]
Xte3 = xte3[:, keep, :]

mean = Xtr3.mean(dim = (0,2), keepdim=True)
std = Xtr3.std(dim=(0,2), keepdim=True).clamp(min = 1e-6)

n_xtr3 = (Xtr3 - mean) / std
n_xte3 = (Xte3 - mean) / std

test_loader_normalised = DataLoader(TensorDataset(n_xte3, yte3), batch_size=64, shuffle=False)
train_loader_normalised = DataLoader(TensorDataset(n_xtr3, ytr3), batch_size=64, shuffle=True)

print("Train:", n_xtr3.shape, n_xtr3.mean().item(), n_xtr3.std().item())
print("Test:", n_xte3.shape)

Train: torch.Size([4107, 15, 100]) 2.4209175535361283e-06 0.9999988675117493
Test: torch.Size([1384, 15, 100])


In [45]:
#training again :)

model3 = CNN(in_channel=15, n_cls=4)
optimizer_3 = torch.optim.Adam(model3.parameters(), lr = 0.001)
criterion_3 = nn.CrossEntropyLoss()

epochs=20
for epoch in range(epochs):

    avg_loss_3 = one_epoch(model3, train_loader_normalised, criterion_3, optimizer_3)

    print(f"epoch:{epoch+1} / {epochs} | loss: {avg_loss_3}")

epoch:1 / 20 | loss: 1.2878587857516455
epoch:2 / 20 | loss: 1.1825282131635797
epoch:3 / 20 | loss: 1.1517080036950047
epoch:4 / 20 | loss: 1.1224517237974658
epoch:5 / 20 | loss: 1.1230975439405104
epoch:6 / 20 | loss: 1.1108535007313565
epoch:7 / 20 | loss: 1.0996502931024548
epoch:8 / 20 | loss: 1.0894576350296388
epoch:9 / 20 | loss: 1.0765218252911601
epoch:10 / 20 | loss: 1.0741628955569846
epoch:11 / 20 | loss: 1.0591042098959549
epoch:12 / 20 | loss: 1.0629819634769384
epoch:13 / 20 | loss: 1.0538669287911697
epoch:14 / 20 | loss: 1.0561998239944008
epoch:15 / 20 | loss: 1.0500921860848837
epoch:16 / 20 | loss: 1.0423461050919904
epoch:17 / 20 | loss: 1.0442499362090105
epoch:18 / 20 | loss: 1.0390842324020741
epoch:19 / 20 | loss: 1.0479536424276457
epoch:20 / 20 | loss: 1.040907374228298


In [46]:
tloss_3, acc_3 = evaluate(model3, test_loader_normalised, criterion3)
print(f"loss: {tloss_3}, acc: {acc_3}")

loss: 1.2659873252659175, acc: 0.34826589595375723


In [47]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import torch

model3.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader_normalised:
        logits = model3(xb)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3]))

print("\nClassification report:")
print(classification_report(
    all_labels, all_preds,
    labels=[0, 1, 2, 3],
    target_names=["Cooling", "Brake", "Engine", "Tyre"],
    zero_division=0
))

Confusion matrix:
[[ 83 152  79 148]
 [ 20 136   9  39]
 [ 51  19 173  69]
 [ 20 295   1  90]]

Classification report:
              precision    recall  f1-score   support

     Cooling       0.48      0.18      0.26       462
       Brake       0.23      0.67      0.34       204
      Engine       0.66      0.55      0.60       312
        Tyre       0.26      0.22      0.24       406

    accuracy                           0.35      1384
   macro avg       0.41      0.41      0.36      1384
weighted avg       0.42      0.35      0.34      1384



In [48]:
print("Train labels:", torch.bincount(ytr3))
print("Test labels:", torch.bincount(yte3))
print("Input channels:", len(keep))
print("Channels kept:", [signal_cols[i] for i in keep])

Train labels: tensor([ 949, 1139, 1056,  963])
Test labels: tensor([462, 204, 312, 406])
Input channels: 15
Channels kept: ['speed', 'rpm', 'engine_temp', 'brake_FL', 'brake_FR', 'brake_RL', 'brake_RR', 'tyre_FL', 'tyre_FR', 'tyre_RL', 'tyre_RR', 'ws_FL', 'ws_FR', 'ws_RL', 'ws_RR']


In [ ]:
#checking for data leakage: no leakage...
train_runs = set(vrids[tr_id])
test_runs = set(vrids[te_id])

print("Train runs:", len(train_runs))
print("Test runs:", len(test_runs))
print("Overlapping runs:", train_runs & test_runs)

Train runs: 120
Test runs: 40
Overlapping runs: set()


In [50]:
#faults represented by each split
print("Training runs per fault:")
print(mdf[mdf["run_id"].isin(train_runs)]["fault"].value_counts())

print("\nTesting runs per fault:")
print(mdf[mdf["run_id"].isin(test_runs)]["fault"].value_counts())

Training runs per fault:
fault
brake_failure       34
engine_failure      31
tyre_degradation    28
cooling_failure     27
Name: count, dtype: int64

Testing runs per fault:
fault
cooling_failure     13
tyre_degradation    12
engine_failure       9
brake_failure        6
Name: count, dtype: int64


try4- updating the splits.

In [52]:
#splitting data again for fresh xte/xtr and y...:
from sklearn.model_selection import StratifiedGroupKFold

keep = [i for i, col in enumerate(signal_cols)
        if not col.startswith("slip_")]

cv = StratifiedGroupKFold(n_splits=5 #before it was 4
                         , shuffle=True, random_state=1)
for fold, (tr_id, te_id) in enumerate(cv.split(X, Y, groups=vrids), start=1):
    xtr4, xte4 = X[tr_id][:, keep, :], X[te_id][:, keep, :]
    ytr4, yte4 = Y[tr_id], Y[te_id]
    
xtr4.shape, xte4.shape, ytr4.shape, yte4.shape

(torch.Size([4386, 15, 100]),
 torch.Size([1105, 15, 100]),
 torch.Size([4386]),
 torch.Size([1105]))

In [54]:
mean = xtr4.mean(dim = (0,2), keepdim=True)
std = xtr4.std(dim=(0,2), keepdim=True).clamp(min = 1e-6)

n_xtr4 = (xtr4 - mean) / std
n_xte4 = (xte4 - mean) / std

test_loader_normalised4 = DataLoader(TensorDataset(n_xte4, yte4), batch_size=64, shuffle=False)
train_loader_normalised4 = DataLoader(TensorDataset(n_xtr4, ytr4), batch_size=64, shuffle=True)

print("Train:", n_xtr4.shape, n_xtr4.mean().item(), n_xtr4.std().item())
print("Test:", n_xte4.shape)

Train: torch.Size([4386, 15, 100]) 1.8786099644785281e-06 0.999998927116394
Test: torch.Size([1105, 15, 100])


In [55]:
#again, training

model4 = CNN(in_channel=15, n_cls=4)
optimizer4 = torch.optim.Adam(model4.parameters(), lr=0.001)
criterion4 = nn.CrossEntropyLoss()

epochs = 20

for epoch in range(epochs):
    train_loss4 = one_epoch(
        model4,
        train_loader_normalised4,
        criterion4,
        optimizer4
    )
    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss4:.4f}")

test_loss4, test_acc4 = evaluate(
    model4, test_loader_normalised4, criterion4)

print(f"loss4: {test_loss4}, accr4: {test_acc4}")

Epoch 1/20 | Loss: 1.2963
Epoch 2/20 | Loss: 1.1930
Epoch 3/20 | Loss: 1.1643
Epoch 4/20 | Loss: 1.1425
Epoch 5/20 | Loss: 1.1374
Epoch 6/20 | Loss: 1.1125
Epoch 7/20 | Loss: 1.1034
Epoch 8/20 | Loss: 1.1114
Epoch 9/20 | Loss: 1.1090
Epoch 10/20 | Loss: 1.0949
Epoch 11/20 | Loss: 1.0867
Epoch 12/20 | Loss: 1.0845
Epoch 13/20 | Loss: 1.0682
Epoch 14/20 | Loss: 1.0718
Epoch 15/20 | Loss: 1.0681
Epoch 16/20 | Loss: 1.0633
Epoch 17/20 | Loss: 1.0609
Epoch 18/20 | Loss: 1.0528
Epoch 19/20 | Loss: 1.0618
Epoch 20/20 | Loss: 1.0494
loss4: 1.1770779842704668, accr4: 0.3592760180995475


In [56]:

from sklearn.metrics import confusion_matrix, classification_report
import torch

model4.eval()
all_preds4, all_labels4 = [], []

with torch.no_grad():
    for xb, yb in test_loader_normalised4:
        preds = model4(xb).argmax(dim=1)
        all_preds4.extend(preds.cpu().numpy())
        all_labels4.extend(yb.cpu().numpy())

print(confusion_matrix(all_labels4, all_preds4, labels=[0, 1, 2, 3]))

print(classification_report(
    all_labels4,
    all_preds4,
    labels=[0, 1, 2, 3],
    target_names=["Cooling", "Brake", "Engine", "Tyre"],
    zero_division=0
))


[[ 89  30 134 108]
 [ 46  55  21 182]
 [ 21   5 136  37]
 [ 13  91  20 117]]
              precision    recall  f1-score   support

     Cooling       0.53      0.25      0.34       361
       Brake       0.30      0.18      0.23       304
      Engine       0.44      0.68      0.53       199
        Tyre       0.26      0.49      0.34       241

    accuracy                           0.36      1105
   macro avg       0.38      0.40      0.36      1105
weighted avg       0.39      0.36      0.34      1105

